### Phase Resetting Analysis ###

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as sps
from scipy.signal import butter, sosfiltfilt
import numpy as np
import scipy.stats as ss
import seaborn as sns
from scipy import stats
import numpy as np
import os
import itertools
import math
from scipy.stats import norm, pearsonr, circmean
import random
import glob
import matplotlib.patches as mpatches
sns.set_theme()
from scipy.ndimage import gaussian_filter1d
%matplotlib widget
%matplotlib inline

In [ ]:
def compute_phase_angles(signal, fs):
    # extract phase angles for a given input signal using hilbert transform
    # returns array of size (n_bands, n_trials, len_signal)
    # signal can be a list of signals from n trials where n = # of trials or a single signal
    frequency_bands = {
    'delta': (1, 3),
    'theta': (3, 8),
    'alpha': (8, 12),
    'beta': (12, 30),
    'low gamma': (30, 50),
    'high gamma': (70, 150)
    }
    phase_angles = []
    complex_phase_angles = []
    
    for band, (low_freq, high_freq) in frequency_bands.items():
        filtered = butter_bandpass_filter(signal, low_freq, high_freq, 1024, order=2)
        phase = np.angle(sps.hilbert(filtered))
        complex_phase = np.exp(1j * phase) # convert to complex phase
        phase_angles.append(phase)
        complex_phase_angles.append(complex_phase)
        
    return phase_angles, complex_phase_angles

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = sps.butter(order, [low, high], btype='bandpass')
    return b, a


def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    # y = sps.lfilter(b, a, data)
    y = sps.filtfilt(b, a, data)
    return y

# Rayleigh Z: https://pmc.ncbi.nlm.nih.gov/articles/PMC8634571/#sec1
def rayleigh_z(phase):
    n_trials = phase.shape[0]
    r = np.abs(np.mean(np.exp(1j * phase), axis=0))
    return n_trials * (r ** 2)

def circular_mean(phase):
    return np.angle(np.mean(np.exp(1j * phase), axis=0))

def get_clusters(sig):
    clusters = []
    start = None

    for i, v in enumerate(sig):
        if v and start is None:
            start = i
        elif not v and start is not None:
            clusters.append((start, i))
            start = None

    if start is not None:
        clusters.append((start, len(sig)))

    return clusters

def circular_shift_perm(phase, n_perm=1000):
    n_trials, n_time = phase.shape

    obs_z = rayleigh_z(phase)
    null_z = np.zeros((n_perm, n_time))

    for i in range(n_perm):
        shifted = np.zeros_like(phase)

        for tr in range(n_trials):
            shift = np.random.randint(n_time)
            shifted[tr] = np.roll(phase[tr], shift)

        null_z[i] = rayleigh_z(shifted)

    p = (np.sum(null_z >= obs_z, axis=0) + 1) / (n_perm + 1)

    return obs_z, p, null_z

def simulate_phase(n_trials, frequency, time, cue_time=0, reset_strength=0, reset_center=0, reset_duration=0.3,
    rng=None):

    if rng is None:
        rng = np.random.default_rng()

    phase_offset = rng.uniform(-np.pi, np.pi, n_trials) # random phase offset for each tr

    # oscillatory phase
    phase = (2 * np.pi * frequency * time[None, :] + phase_offset[:, None])

    # cue-locked phase reset
    if reset_strength > 0:
        # trial-specific phase offsets sampled from von Mises
        reset_offsets = rng.vonmises(mu=reset_center, kappa=reset_strength, size=n_trials)
        relative_time = time - cue_time
        ramp = np.clip(relative_time / 0.05, 0, 1)
        decay = np.exp(-np.maximum(relative_time, 0) / reset_duration)
        reset_weight = ramp * decay
        reset_weight[relative_time < 0] = 0

        # circular interpolation between random and reset phases
        phase_difference = np.angle(np.exp(1j * (reset_offsets[:, None] - phase_offset[:, None])))
        phase += (phase_difference * reset_weight[None, :])

    return phase

# simulation functions
def simulate_neural_signal(n_trials, time, condition, roi, contact, subject, rng):
    signal = np.zeros((n_trials, len(time)))

    # condition-specific phase-reset strengths
    reset_strengths = {
        'correct': {
            'CMT': 1.5,
            'IFG': 3.0,
            'MTG': 0.5
        },
        'incorrect': {
            'CMT': 0.3,
            'IFG': 0.3,
            'MTG': 0.2
        }
    }

    for band, (fmin, fmax) in frequency_bands.items():
        frequency = np.mean([fmin, fmax])
        frequency += rng.normal(0, 0.1) # add small contact-specific frequency variability
        amplitude = rng.uniform(0.5, 1.5) # amp variability
        kappa = reset_strengths[condition][roi]

        # simulate phase
        phase = simulate_phase(
            n_trials=n_trials,
            frequency=frequency,
            time=time,
            reset_strength=kappa,
            reset_duration=0.25,
            rng=rng
        )

        oscillation = amplitude * np.cos(phase)
        signal += oscillation

    # add broadband noise
    noise = rng.normal(0, 1.5, size=(n_trials, len(time)))
    signal += noise
    signal *= rng.uniform(0.8, 1.2)

    return signal

### Generate simulated data

In [ ]:
# parameters
rng = np.random.default_rng(42)
subjid_list = ['e0020JA', 'e0022ZG', 'e0027VI','e0028ZM', 'e0030SG', 'e0034EL', 'e0041VV']
roi_list = ['CMT', 'IFG', 'MTG']
fs = 1024

n_contacts = {
    'CMT': 3,
    'IFG': 4,
    'MTG': 4
}

n_trials = {
    'correct': 60,
    'incorrect': 40
}

n_samples = 2048 # trial duration
time = np.arange(n_samples) / fs - 1
dt_trial = ['150']

frequency_bands = {
    'delta': (1, 3),
    'theta': (3, 8),
    'alpha': (8, 12),
    'beta': (12, 30),
    'low gamma': (30, 50),
    'high gamma': (70, 150)
}

# run simulated data generation
sig_all = {}
for subjid in subjid_list:
    sig_all[subjid] = {}
    
    for roi in roi_list:
        sig_all[subjid][roi] = []
        
        for contact in range(n_contacts[roi]):
            contact_data = {}
            
            for err in ['correct', 'incorrect']:
                contact_data[err] = {}
                
                for dt in dt_trial:
                    ntrials = n_trials[err]
                    simulated_signal = simulate_neural_signal(
                        n_trials=ntrials,
                        time=time,
                        condition=err,
                        roi=roi,
                        contact=contact,
                        subject=subjid,
                        rng=rng
                    )

                    contact_data[err][dt] = {'cue': simulated_signal}
            sig_all[subjid][roi].append(contact_data)

In [ ]:
# assumes that you have a dictionary with the preprocessed neural data with the following structure:
# (n_subjects, n_rois, n_contacts, accuracy, dt, cue, n_trials)

subjid_list = ['e0020JA', 'e0022ZG', 'e0027VI', 'e0028ZM', 'e0030SG', 'e0034EL', 'e0041VV']

# band of interest
fmin = 1
fmax = 150
band = 'all'
errs = ['correct','incorrect']

# calculate instantaneous phase by band for each contact using hilbert transform
phase_all = {subj: {roi: [] for roi in sig_all[subjid].keys()} for subj in subjid_list} # shape for each key is (n_bands, n_trials, len_signal)
itpc_all = {subj: {roi: [] for roi in  sig_all[subj].keys()} for subj in subjid_list} # shape for each key is (n_bands, len_signal)

for subjid in subjid_list:
    for roi in list(sig_all[subjid].keys()):
        for c,contact in enumerate(sig_all[subjid][roi]):
            phase_list = {err: {dt: [] for dt in dt_trial} for err in errs}
            complex_list = {err: {dt: [] for dt in dt_trial} for err in errs}
            for err in errs:
                for dt in dt_trial:
                    if len(sig_all[subjid][roi][c][err][dt]['cue'])>0:
                        phase, complex_phase = compute_phase_angles(sig_all[subjid][roi][c][err][dt]['cue'], 1024)
                        itpc = np.abs(np.mean(complex_phase, axis=1)) # results in array of itpc values of shape (n_bands, len_signal)
                        phase_list[err][dt]=phase
                        complex_list[err][dt]=itpc

            # append list for each contact
            phase_all[subjid][roi].append(phase_list)
            itpc_all[subjid][roi].append(complex_list)

In [ ]:
# save phase data
fs = 1024
win = fs
dt = '150'
band = 1
hem = 'R'

if band==1:
    band_name='theta'
elif band==2:
    band_name='alpha'
elif band==3:
    band_name='beta'
elif band==4:
    band_name='lowgamma'
elif band==5:
    band_name='highgamma'

for subjid in subjid_list:
    for roi in sig_all[subjid].keys():
        for e, err in enumerate(errs):
            contact_data = []
            for c, contact in enumerate(phase_all[subjid][roi]):
                phase = phase_all[subjid][roi][c][err][dt][band]
                df = pd.DataFrame(phase)
                if not df.empty:
                    df.to_excel(f'/Volumes/LAB/data/{hem}_{roi}_{c}_{dt}_{subjid}_{err}_{band_name}.xlsx', index=False, header=False)

In [ ]:
# Running Rayleigh Z-statistic calculation

# parameters
base_path = '/Volumes/LAB/data/'
save_path = '/Volumes/LAB/data/rayleigh'
os.makedirs(save_path, exist_ok=True)
bands = ['theta']
regions = ['IFG']
hems = ['R'] # L, R
errs = ['correct', 'incorrect']
colors = {'correct': 'green', 'incorrect': 'red'}
fs = 1024

# 500ms around cue
t = np.linspace(-1000, 1000, 2048)

# main
for subjid in subjid_list:
    for hem in hems:
        for band in bands:
            for region in regions:

                files_by_err = {}
                for err in errs:
                    pattern = os.path.join(base_path,
                        f'{hem}_{region}_*_150_{subjid}_{err}_{band}.xlsx'
                    )
                    files = glob.glob(pattern)
                    files_by_err[err] = files
                max_files = max(len(files_by_err['correct']), len(files_by_err['incorrect']))

                if max_files == 0:
                    continue

                fig, axes = plt.subplots(1, max_files, figsize=(2.5 * max_files, 4), sharex=True)
                
                for f in range(max_files):
                    if max_files == 1:
                        axes = [axes]
                    else:
                        ax = axes[f]
                    for err in errs:
                        files = files_by_err[err]

                        if f >= len(files):
                            continue

                        file = files[f]
                        df = pd.read_excel(file, header=None)
                        phase = df.values[:]
                        phase_wrapped = np.angle(np.exp(1j * phase))

                        # Rayleigh Z
                        rz = rayleigh_z(phase_wrapped)
                        if err == "correct":
                            rz_correct = rz.copy()
                        else:
                            rz_incorrect = rz.copy()
                        ax.plot(t, rz, color=colors[err], linewidth=1.5, alpha=0.9, label=err)

                        # significance
                        _, p, _ = circular_shift_perm(phase_wrapped) 
                        sig = p < 0.05
                        clusters = get_clusters(sig)

                        for c in clusters:
                            ax.axvspan(t[c[0]], t[c[1]-1], color=colors[err], alpha=0.15)
                    
                    # determine significant periods
                    for err in errs:
                        phase_err = pd.read_excel(files_by_err[err][f], header=None).values[:]
                        phase_err = np.angle(np.exp(1j * phase_err))
                        _, p_err, _ = circular_shift_perm(phase_err)
    
                        sig = p_err < 0.05
                        print(f'{subjid} {hem} {band} {region} #{str(f)}{err}: p={p_err}')
                                            
                    # formatting
                    ax.axvline(0, linestyle='--', color='black')
                    ax.set_ylim((0,5)) 

                    ax.set_title(f'Contact {f+1}')
                    if f == 0:
                        ax.set_ylabel('Rayleigh Z')

                    ax.set_xlabel('Time (ms)')
                    ax.legend()

                plt.tight_layout()
                plt.savefig(f'{save_path}/{hem}_{region}_{band}_{subjid}.png', dpi=300)
                plt.close()

In [ ]:
# plot correct and incorrect ITPC across time 
fs = 1024
win = fs
dt = '150'
t = np.linspace(int(-win * 1000 / fs), int(win * 1000 / fs), 2 * fs)
bands = ['delta', 'theta', 'alpha', 'beta']
colors=['purple', 'blue', 'red', 'green']
line = ['-', '--', '-.']

for subjid in subjid_list:
    for roi in sig_all[subjid].keys():
        fig, ax = plt.subplots(4, 1, figsize=(8, 12), dpi=300, sharex=True)
        plot_dict = {band: {err:[] for err in errs} for band in bands}

        for b, band in enumerate(bands):
            for e, err in enumerate(errs):
                contact_data = []
                for c, contact in enumerate(sig_all[subjid][roi]):
                    itpc = itpc_all[subjid][roi][c][err][dt][b, :]
                    contact_data.append(itpc)

                contact_data = np.array(contact_data)  # shape: (n_contacts, n_timepoints)
                contact_avg = np.mean(contact_data, axis=0)
                s = ss.sem(contact_data, axis=0)
                plot_dict[band][err] = contact_avg
                
                ax[b].plot(t, contact_avg, color=colors[b], linestyle=line[e], label=f'{err}')
                ax[b].fill_between(t, contact_avg - s, contact_avg + s, alpha=0.25, color=colors[b])
        
            ax[b].set_xlim((-1000, 1000))
            ax[b].set_xlabel('Time (ms)', fontsize=16)
            ax[b].set_ylim((0,1))
            ax[b].set_title(f'{roi} -  {band} - 150ms', fontsize=16)
            ax[b].legend()
        plt.savefig(f'/Volumes/LAB/data/itpc/{roi}_{dt}_{subjid}_{hem}_summary.jpg')
        plt.close()

        # save file
        df = pd.DataFrame(plot_dict)
        df.to_excel(f'/Volumes/LAB/data/itpc/data/{roi}_{subjid}_{hem}.xlsx', index=False, header=False)